# 01 · Collect

Downloads raw Steam responses into `data/raw` (reviews, news, store details, online players). Settings are in `configs/config.yaml`.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import logging

from src.collect.run import collect, plan
from src.config import CONFIG
from src.db import read
from src.db.connection import init_db

logging.basicConfig(level=logging.WARNING, format="%(asctime)s %(levelname)s %(name)s: %(message)s")
init_db()

In [ ]:
SOURCES = "all"   # or e.g. ["reviews"]; all = store_details, reviews, news, player_counts
APPIDS = [570, 553850]   # any Steam app IDs; None = all games in configs/games.yaml (new IDs are added there)
GROUPS = {}   # optional, e.g. {367520: "indie"}: sets the game's group in configs/games.yaml
PER_LANG = 0  # reviews per game x language; None = config, 0 = all reviews

jobs = plan(SOURCES, APPIDS, PER_LANG, GROUPS)
requests = jobs["requests"].sum()
hours = requests / CONFIG.http.host_rps["api.steampowered.com"] / 3600
print(f"{len(jobs)} jobs, {requests:,} requests, about {hours:.1f} h")
jobs

## Run

In [ ]:
result = collect(jobs)
result.groupby(["source", "status"]).size()

In [ ]:
# Failed jobs (re-run this notebook to retry them)
result[result["status"] == "failed"]

## Review download state

One row per game and language, kept across runs.

In [ ]:
read.review_collection()